In [1]:
import json
import time
import pandas as pd
from nba_api.stats.endpoints import (
    shotchartdetail,
    leaguedashplayerstats,
    leaguedashteamstats,
    commonplayerinfo,
)

/Users/noahzameerlee/Documents/stat330/case_studies/env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
SEASON = "2024-25"
GAME_TYPES = ["Regular Season", "Playoffs"]

# shot data

def fetch_shots(season: str, season_type: str) -> pd.DataFrame:
    print(f"  Fetching shots: {season} {season_type}…")
    shot_json = shotchartdetail.ShotChartDetail(
        team_id=0,
        player_id=0,
        context_measure_simple="FGA",
        season_nullable=season,
        season_type_all_star=season_type,
    )
    data = json.loads(shot_json.get_json())
    result = data["resultSets"][0]
    df = pd.DataFrame(result["rowSet"], columns=result["headers"])
    df["SEASON"] = season
    df["GAME_TYPE"] = season_type
    print(f"    → {len(df):,} rows")
    return df


print("=== Fetching shot data ===")
shots_parts = [fetch_shots(SEASON, gt) for gt in GAME_TYPES]
combined_shots = pd.concat(shots_parts, ignore_index=True)
print(f"Total shots: {len(combined_shots):,}\n")


=== Fetching shot data ===
  Fetching shots: 2024-25 Regular Season…
    → 102,400 rows
  Fetching shots: 2024-25 Playoffs…
    → 14,377 rows
Total shots: 116,777



In [3]:
#player data

def fetch_player_stats(season: str, season_type: str) -> pd.DataFrame:
    print(f"  Fetching player stats: {season} {season_type}…")
    df = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star=season_type,
        per_mode_detailed="PerGame",
    ).get_data_frames()[0]
    df["SEASON"] = season
    df["GAME_TYPE"] = season_type
    return df


def get_player_bio(player_id: int) -> pd.Series:
    info = commonplayerinfo.CommonPlayerInfo(player_id=player_id).get_data_frames()[0]
    return info[["PERSON_ID", "HEIGHT", "WEIGHT", "POSITION"]].iloc[0]


print("=== Fetching player stats ===")
player_parts = [fetch_player_stats(SEASON, gt) for gt in GAME_TYPES]
all_player_stats = pd.concat(player_parts, ignore_index=True)

# Bio data (fetched once per unique player — same bio regardless of game type)
print("  Fetching player bios…")
bios = []
unique_ids = all_player_stats["PLAYER_ID"].unique()
for i, pid in enumerate(unique_ids, 1):
    if i % 50 == 0:
        print(f"    {i}/{len(unique_ids)}")
    bios.append(get_player_bio(pid))
    time.sleep(0.6)

bio_df = pd.DataFrame(bios).rename(columns={"PERSON_ID": "PLAYER_ID"})
all_player_stats = all_player_stats.merge(bio_df, on="PLAYER_ID", how="left")

PLAYER_COLS = [
    "PLAYER_ID", "SEASON", "GAME_TYPE", "PLAYER_NAME", "TEAM_ABBREVIATION",
    "AGE", "GP", "MIN", "PTS", "FG_PCT", "FG3_PCT", "PLUS_MINUS",
    "HEIGHT", "WEIGHT", "POSITION",
]
player_stats_slim = all_player_stats[PLAYER_COLS]

combined_shots = combined_shots.merge(
    player_stats_slim,
    on=["PLAYER_ID", "SEASON", "GAME_TYPE"],
    how="left",
)
combined_shots = combined_shots.rename(columns={"TEAM_ID_x": "TEAM_ID"})
print(f"After player merge: {len(combined_shots):,} rows\n")



=== Fetching player stats ===
  Fetching player stats: 2024-25 Regular Season…
  Fetching player stats: 2024-25 Playoffs…
  Fetching player bios…
    50/569
    100/569
    150/569
    200/569
    250/569
    300/569
    350/569
    400/569
    450/569
    500/569
    550/569
After player merge: 116,777 rows



In [4]:
# team stats

def fetch_team_stats(season: str, season_type: str) -> pd.DataFrame:
    print(f"  Fetching team stats: {season} {season_type}…")
    df = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star=season_type,
        per_mode_detailed="PerGame",
    ).get_data_frames()[0]
    df["SEASON"] = season
    df["GAME_TYPE"] = season_type
    return df


print("=== Fetching team stats ===")
team_parts = [fetch_team_stats(SEASON, gt) for gt in GAME_TYPES]
all_teams = pd.concat(team_parts, ignore_index=True)

TEAM_COLS = [
    "TEAM_ID", "SEASON", "GAME_TYPE", "TEAM_NAME",
    "GP", "W", "L", "W_PCT", "PTS", "FG_PCT", "FG3_PCT",
    "REB", "AST", "TOV", "PLUS_MINUS",
]
team_stats_slim = all_teams[TEAM_COLS].copy()

# Prefix team stat columns (keep join keys unprefixed)
team_stats_slim.columns = (
    ["TEAM_ID", "SEASON", "GAME_TYPE"]
    + [f"TEAM_{c}" for c in TEAM_COLS[3:]]
)

combined_shots = combined_shots.merge(
    team_stats_slim,
    on=["TEAM_ID", "SEASON", "GAME_TYPE"],
    how="left",
)
print(f"After team merge: {len(combined_shots):,} rows\n")



=== Fetching team stats ===
  Fetching team stats: 2024-25 Regular Season…
  Fetching team stats: 2024-25 Playoffs…
After team merge: 116,777 rows



In [5]:
combined_shots

,GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME_x,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,...,TEAM_W,TEAM_L,TEAM_W_PCT,TEAM_PTS,TEAM_FG_PCT,TEAM_FG3_PCT,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_PLUS_MINUS
0,Shot Chart Detail,0022400001,7,1642258,Zaccharie Risacher,1610612737,Atlanta Hawks,1,11,43,...,40,42,0.488,118.2,0.472,0.358,44.5,29.6,15.5,-1.1
1,Shot Chart Detail,0022400001,10,1630552,Jalen Johnson,1610612737,Atlanta Hawks,1,11,38,...,40,42,0.488,118.2,0.472,0.358,44.5,29.6,15.5,-1.1
2,Shot Chart Detail,0022400001,12,1628401,Derrick White,1610612738,Boston Celtics,1,11,24,...,61,21,0.744,116.3,0.462,0.368,45.3,26.1,11.9,9.1
3,Shot Chart Detail,0022400001,21,1630552,Jalen Johnson,1610612737,Atlanta Hawks,1,10,50,...,40,42,0.488,118.2,0.472,0.358,44.5,29.6,15.5,-1.1
4,Shot Chart Detail,0022400001,23,1627759,Jaylen Brown,1610612738,Boston Celtics,1,10,35,...,61,21,0.744,116.3,0.462,0.368,45.3,26.1,11.9,9.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116772,Shot Chart Detail,0042400407,705,1631096,Chet Holmgren,1610612760,Oklahoma City Thunder,4,2,24,...,16,7,0.696,114.7,0.456,0.338,43.6,23.0,12.3,8.3
116773,Shot Chart Detail,0042400407,707,1631097,Bennedict Mathurin,1610612754,Indiana Pacers,4,2,16,...,15,8,0.652,114.1,0.484,0.390,40.6,26.4,14.3,2.0
116774,Shot Chart Detail,0042400407,733,1631097,Bennedict Mathurin,1610612754,Indiana Pacers,4,0,56,...,15,8,0.652,114.1,0.484,0.390,40.6,26.4,14.3,2.0
116775,Shot Chart Detail,0042400407,736,1629652,Luguentz Dort,1610612760,Oklahoma City Thunder,4,0,33,...,16,7,0.696,114.7,0.456,0.338,43.6,23.0,12.3,8.3


In [6]:
OUT = "nba_shots_combined.csv"
combined_shots.to_csv(OUT, index=False)
print(f"=== Done! Saved {len(combined_shots):,} rows → {OUT}")
print(f"Columns ({len(combined_shots.columns)}): {list(combined_shots.columns)}")
print("\nGame type breakdown:")
print(combined_shots["GAME_TYPE"].value_counts().to_string())

=== Done! Saved 116,777 rows → nba_shots_combined.csv
Columns (50): ['GRID_TYPE', 'GAME_ID', 'GAME_EVENT_ID', 'PLAYER_ID', 'PLAYER_NAME_x', 'TEAM_ID', 'TEAM_NAME', 'PERIOD', 'MINUTES_REMAINING', 'SECONDS_REMAINING', 'EVENT_TYPE', 'ACTION_TYPE', 'SHOT_TYPE', 'SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA', 'SHOT_ZONE_RANGE', 'SHOT_DISTANCE', 'LOC_X', 'LOC_Y', 'SHOT_ATTEMPTED_FLAG', 'SHOT_MADE_FLAG', 'GAME_DATE', 'HTM', 'VTM', 'SEASON', 'GAME_TYPE', 'PLAYER_NAME_y', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'MIN', 'PTS', 'FG_PCT', 'FG3_PCT', 'PLUS_MINUS', 'HEIGHT', 'WEIGHT', 'POSITION', 'TEAM_TEAM_NAME', 'TEAM_GP', 'TEAM_W', 'TEAM_L', 'TEAM_W_PCT', 'TEAM_PTS', 'TEAM_FG_PCT', 'TEAM_FG3_PCT', 'TEAM_REB', 'TEAM_AST', 'TEAM_TOV', 'TEAM_PLUS_MINUS']

Game type breakdown:
GAME_TYPE
Regular Season    102400
Playoffs           14377
